# Shofly media worker (Colab)
**Manual, non-persistent worker.** Run all cells; it claims READY import jobs, downloads from the *authorized* source, probes with ffprobe,
uploads to the configured storage provider, verifies, and only then reports completion. If Colab disconnects, jobs are reclaimed after their lease expires.
Never paste secrets into cells: add them in Colab **Secrets** (key icon) with these names: `SHOFLY_API_URL`, `WORKER_API_TOKEN`, and for Telegram storage `TELEGRAM_BOT_TOKEN`, `TELEGRAM_STORAGE_CHANNEL_ID`.

## 1. Environment

In [ ]:
import os, subprocess, sys
# Get the worker code: upload the `workers/media-worker` folder to /content, or clone your own repo.
REPO_URL = ""  # e.g. https://github.com/you/shofly.git  (leave empty if you uploaded the folder)
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/shofly"], check=True)
    WORKER_DIR = "/content/shofly/workers/media-worker"
else:
    WORKER_DIR = "/content/media-worker"
assert os.path.isdir(WORKER_DIR), f"worker code not found at {WORKER_DIR}"
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "httpx==0.28.1", "pydantic-settings==2.7.0"], check=True)
sys.path.insert(0, WORKER_DIR)
print("ffprobe:", subprocess.run(["ffprobe", "-version"], capture_output=True, text=True).stdout.splitlines()[:1] or "MISSING (run: !apt-get -qq install -y ffmpeg)")

## 2. Secrets and checks

In [ ]:
from google.colab import userdata
os.environ["API_BASE_URL"] = userdata.get("SHOFLY_API_URL")
os.environ["WORKER_API_TOKEN"] = userdata.get("WORKER_API_TOKEN")
os.environ["PROVIDER"] = "telegram"   # or "local" for a dry run into /content/stored
if os.environ["PROVIDER"] == "telegram":
    os.environ["TELEGRAM_BOT_TOKEN"] = userdata.get("TELEGRAM_BOT_TOKEN")
    os.environ["TELEGRAM_CHAT_ID"] = userdata.get("TELEGRAM_STORAGE_CHANNEL_ID")

import httpx
from shofly_worker.config import WorkerSettings
from shofly_worker.client import BackendClient
from shofly_worker.providers import LocalDirProvider, TelegramProvider
S = WorkerSettings()
client = BackendClient(S.api_base_url, S.worker_api_token, S.worker_id)
print("backend live:", httpx.get(S.api_base_url + "/health/live", timeout=15).json())
print("worker token accepted:", client.ping(), "| worker id:", S.worker_id)
provider = TelegramProvider(S.telegram_bot_token, S.telegram_chat_id, S.max_bytes) if S.provider == "telegram" else LocalDirProvider(S.local_dir)

## 3. Process jobs
Prints one outcome per job (no URLs or tokens). Stop the cell at any time: unfinished jobs are re-queued when their lease expires.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
from shofly_worker.runner import run_until_empty
dl_http = httpx.Client(timeout=60.0)
results = run_until_empty(client, provider, S, dl_http)
from collections import Counter
print("processed:", len(results), dict(Counter(results)))

## 4. Cleanup

In [ ]:
import shutil
shutil.rmtree(S.tmp_dir, ignore_errors=True)
print("temp files removed")